[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_Translation.ipynb)

# Benchmark: Translation

Scores a pretrained translation model's BLEU against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="translation")`.

**Dataset**: [Tatoeba](https://tatoeba.org/) English-French sentence pairs, via
[OPUS](https://opus.nlpl.eu/Tatoeba/en&fr/v2023-04-12/Tatoeba) -- the same crowd-sourced sentence
collection `opus_mt_en_fr` itself was trained on. We sample 200 short sentence pairs to keep this
notebook quick to run.

**Model**: `MarianTransformer.pretrained("opus_mt_en_fr")`, Spark NLP's default English->French
translation model.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import MarianTransformer
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import io
import urllib.request
import zipfile

url = "https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/en-fr.txt.zip"
archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
en_lines = archive.read("Tatoeba.en-fr.en").decode("utf-8").splitlines()
fr_lines = archive.read("Tatoeba.en-fr.fr").decode("utf-8").splitlines()

pairs = [(en, fr) for en, fr in zip(en_lines, fr_lines) if 20 <= len(en) <= 120][:200]
gold_data = spark.createDataFrame(pairs, ["text", "label"])
print(gold_data.count(), "sentence pairs")
gold_data.show(3, truncate=60)

200 sentence pairs
+------------------------------------------------------------+------------------------------------------------------------+
|                                                        text|                                                       label|
+------------------------------------------------------------+------------------------------------------------------------+
|When he asked who had broken the window, all the boys put...|Lorsqu'il a demandé qui avait cassé la fenêtre, tous les ...|
|Then, when he asked who had broken the window, all the bo...|Lorsqu'il a demandé qui avait cassé la fenêtre, tous les ...|
|                                       Let's try something. |                                   Essayons quelque chose ! |
+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows

## 2. Build the pipeline

In [10]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
translator = MarianTransformer.pretrained("opus_mt_en_fr") \
    .setInputCols(["document"]).setOutputCol("translation")

pipeline = Pipeline(stages=[document_assembler, translator])
pipeline_model = pipeline.fit(gold_data)

opus_mt_en_fr download started this may take some time.
Approximate size to download 378.7 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[OK!]

> **Note: `predicted_col` disambiguates same-typed columns.** Both `document` (the raw
> input, echoed by `DocumentAssembler`) and `translation` (the model's actual output) are
> annotator type `document` -- auto-detection can't tell them apart, so we pass `predicted_col`
> explicitly, the same disambiguation used in the `SpellCheck` notebook.

## 3. Run the benchmark

In [13]:
# Both DocumentAssembler and MarianTransformer emit annotatorType "document". Benchmark scores
# the last one -- the translation -- and names it in the report, so you can confirm at a glance
# that the score isn't accidentally measuring the untranslated source text.
report = Benchmark.evaluate(pipeline_model, gold_data, task="translation", label_col="label")
print(report)

translation accuracy (n=200): bleu=0.5460